In [34]:
# import os
# os.kill(os.getpid(), 9)

In [ ]:
import os
os.kill(os.getpid(), 9)

In [1]:
# ============================================================
# PyTorch LSTM Sentiment Analysis using IMDb Dataset
# Google Colab Ready
# ============================================================

# Install the 'datasets' library, which provides easy access to many NLP datasets.
!pip install datasets -q

# ============================================================
# IMPORT LIBRARIES
# ============================================================

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader
# `pad_sequence` is used to make all sequences in a batch the same length by padding.
from torch.nn.utils.rnn import pad_sequence

# Library to load popular datasets easily.
from datasets import load_dataset

# `Counter` is used for counting hashable objects (like words).
from collections import Counter

# Regular expression operations for text cleaning.
import re

# ============================================================
# DEVICE CONFIGURATION
# ============================================================

# Determine if a CUDA-enabled GPU is available and use it; otherwise, fall back to CPU.
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using Device:", device)

# ============================================================
# LOAD IMDb DATASET
# ============================================================

# Load the IMDb dataset, which is commonly used for sentiment analysis tasks.
dataset = load_dataset("imdb")

# Extract the training and testing splits of the dataset.
train_data = dataset["train"]
test_data = dataset["test"]

# Print the first example from the training data to inspect its structure (text and label).
print(train_data[0])

# ============================================================
# TEXT CLEANING FUNCTION
# ============================================================

def clean_text(text):
    # Convert all text to lowercase to ensure consistency and reduce vocabulary size.
    text = text.lower()
    # Remove HTML tags (e.g., '<br />') using regular expressions.
    text = re.sub(r"<.*?>", "", text)
    # Remove any characters that are not alphabetic letters or spaces.
    text = re.sub(r"[^a-zA-Z\s]", "", text)
    return text

# ============================================================
# TOKENIZATION FUNCTION
# ============================================================

def tokenize(text):
    # Clean the input text and then split it into individual words (tokens).
    return clean_text(text).split()

# ============================================================
# BUILD VOCABULARY
# ============================================================

# Initialize a Counter to store word frequencies.
counter = Counter()

# Iterate through each sample in the training data.
for sample in train_data:
    # Tokenize the text of each sample and update the word counts in the counter.
    counter.update(tokenize(sample["text"]))

# ============================================================
# CREATE VOCABULARY MAPPING
# ============================================================

vocab = {}

# Add special tokens for padding and unknown words.
vocab["<PAD>"] = 0  # Padding token, typically mapped to index 0.
vocab["<UNK>"] = 1  # Unknown word token, for words not in our vocabulary.

# Start assigning indices for regular words from 2, as 0 and 1 are reserved.
index = 2

# Iterate through words and their frequencies from the counter.
for word, freq in counter.items():
    # Only include words that appear at least 5 times to filter out rare words and reduce vocabulary size.
    if freq >= 5:
        vocab[word] = index
        index += 1

# ============================================================
# VOCABULARY SIZE CALCULATION
# ============================================================

# The vocabulary size is the highest assigned index + 1, as indices are 0-based.
vocab_size = max(vocab.values()) + 1

print("Vocabulary Size:", vocab_size)
print("Maximum Index:", max(vocab.values()))

# ============================================================
# ENCODING FUNCTION
# ============================================================

def encode(text):
    # Tokenize the input text using the defined tokenize function.
    tokens = tokenize(text)
    # Convert each token into its numerical index from the vocabulary.
    # If a token is not found in the vocab, use the index for the unknown token ('<UNK>').
    return [
        vocab.get(token, vocab["<UNK>"])
        for token in tokens
    ]

# ============================================================
# CUSTOM DATASET CLASS (IMDBDataset)
# ============================================================

class IMDBDataset(Dataset):
    # Custom Dataset class to handle the IMDb movie review data.
    def __init__(self, data):
        # Encode each text review into a list of numerical indices and convert to a PyTorch tensor.
        self.texts = [
            torch.tensor(
                encode(item["text"]),
                dtype=torch.long # Use long dtype for indices in embedding layers.
            )
            for item in data
        ]
        # Convert labels (0 or 1) into PyTorch float32 tensors.
        self.labels = [
            torch.tensor(
                item["label"]),
                dtype=torch.float32 # Use float32 for labels, especially for BCEWithLogitsLoss.
            )
            for item in data
        ]

    def __len__(self):
        # Return the total number of samples in the dataset.
        return len(self.labels)

    def __getitem__(self, idx):
        # Retrieve a single encoded text tensor and its corresponding label by index.
        return self.texts[idx], self.labels[idx]

# ============================================================
# PADDING FUNCTION FOR DATALOADER (collate_batch)
# ============================================================

def collate_batch(batch):
    # This function is used by the DataLoader to process a list of individual samples into a batch.

    # Separate the text tensors and label tensors from the batch.
    texts = [item[0] for item in batch]
    labels = [item[1] for item in batch]

    # Pad sequences to ensure all text tensors in the batch have the same length.
    # `batch_first=True` means the batch dimension comes first.
    # `padding_value=vocab["<PAD>"]` fills shorter sequences with the padding token index.
    texts = pad_sequence(
        texts,
        batch_first=True,
        padding_value=vocab["<PAD>"]
    )

    # Stack the individual label tensors into a single batch tensor.
    labels = torch.stack(labels)

    return texts, labels

# ============================================================
# CREATE DATASET INSTANCES
# ============================================================

# Create instances of the custom dataset for training and testing data.
train_dataset = IMDBDataset(train_data)
test_dataset = IMDBDataset(test_data)

# ============================================================
# CREATE DATALOADERS
# ============================================================

# Create DataLoader for the training set.
# `batch_size`: Number of samples per batch.
# `shuffle=True`: Shuffles the training data each epoch for better generalization.
# `collate_fn`: Specifies how to combine individual samples into a batch.
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    collate_fn=collate_batch
)

# Create DataLoader for the testing set.
# `shuffle=False`: No need to shuffle test data.
test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    collate_fn=collate_batch
)

# ============================================================
# LSTM MODEL ARCHITECTURE (SentimentLSTM)
# ============================================================

class SentimentLSTM(nn.Module):
    # Defines the Long Short-Term Memory (LSTM) neural network for sentiment classification.
    def __init__(
        self,
        vocab_size,
        embedding_dim=128, # Dimension of the word embeddings.
        hidden_dim=128     # Dimension of the LSTM's hidden state.
    ):

        super().__init__()

        # Embedding Layer: Converts word indices into dense vectors.
        # `vocab_size`: Total number of unique words in the vocabulary.
        # `embedding_dim`: The size of each word vector.
        # `padding_idx=0`: Specifies that the padding token (index 0) should not contribute to the gradients.
        self.embedding = nn.Embedding(
            vocab_size,
            embedding_dim,
            padding_idx=0
        )

        # LSTM Layer: Processes sequential data.
        # `embedding_dim`: Input size, which is the output of the embedding layer.
        # `hidden_dim`: The number of features in the hidden state `h`.
        # `batch_first=True`: Input and output tensors are provided as (batch, seq, feature).
        self.lstm = nn.LSTM(
            embedding_dim,
            hidden_dim,
            batch_first=True
        )

        # Fully Connected Layer: Maps the output of the LSTM to the final prediction.
        # `hidden_dim`: Input size, which is the last hidden state of the LSTM.
        # `1`: Output size, as we are doing binary classification (e.g., positive/negative).
        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        # Forward pass definition for the model.

        # Pass input indices through the embedding layer.
        embedded = self.embedding(x)

        # Pass the embedded sequences through the LSTM layer.
        # `output`: Contains the output features (h_t) from the last layer for each t.
        # `(hidden, cell)`: Contains the final hidden state and cell state for each layer.
        output, (hidden, cell) = self.lstm(embedded)

        # We take the hidden state from the last LSTM layer (`hidden[-1]`) for classification.
        hidden = hidden[-1]

        # Pass the final hidden state through the fully connected layer.
        out = self.fc(hidden)

        # Remove the singleton dimension (batch_size, 1) to (batch_size,) for BCEWithLogitsLoss.
        return out.squeeze(1)

# ============================================================
# INITIALIZE MODEL, LOSS FUNCTION, AND OPTIMIZER
# ============================================================

# Create an instance of the SentimentLSTM model and move it to the specified device (CPU/GPU).
model = SentimentLSTM(vocab_size).to(device)

# Define the loss function: Binary Cross-Entropy with Logits Loss.
# This combines sigmoid activation and BCE loss, providing numerical stability.
criterion = nn.BCEWithLogitsLoss()

# Define the optimizer: Adam optimizer is used to update model parameters.
# `model.parameters()`: All learnable parameters of the model.
# `lr`: Learning rate, controls the step size during optimization.
optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

# ============================================================
# TRAINING LOOP
# ============================================================

# Set the number of training epochs.
epochs = 5

# Iterate through the specified number of epochs.
for epoch in range(epochs):

    # Set the model to training mode.
    model.train()

    # Initialize total loss for the current epoch.
    total_loss = 0

    # Iterate through batches of data from the training data loader.
    for texts, labels in train_loader:

        # Move input texts and labels to the designated device (CPU/GPU).
        texts = texts.to(device)
        labels = labels.to(device)

        # Forward pass: Get predictions from the model.
        predictions = model(texts)

        # Calculate the loss between predictions and actual labels.
        loss = criterion(predictions, labels)

        # Zero out the gradients from the previous iteration.
        optimizer.zero_grad()

        # Backward pass: Compute gradients of the loss with respect to model parameters.
        loss.backward()

        # Optimizer step: Update model parameters using the computed gradients.
        optimizer.step()

        # Accumulate the batch loss.
        total_loss += loss.item()

    # Calculate the average loss for the epoch.
    avg_loss = total_loss / len(train_loader)

    # Print training progress for the current epoch.
    print(
        f"Epoch [{epoch+1}/{epochs}] "
        f"Loss: {avg_loss:.4f}"
    )

# ============================================================
# EVALUATION FUNCTION
# ============================================================

def evaluate(model, loader):
    # Function to evaluate the model's performance on a given dataset.

    # Set the model to evaluation mode (disables dropout, batchnorm updates, etc.).
    model.eval()

    correct = 0 # Counter for correctly predicted samples.
    total = 0   # Counter for total samples evaluated.

    # Disable gradient calculations during evaluation to save memory and speed up computation.
    with torch.no_grad():

        # Iterate through batches of data from the provided data loader.
        for texts, labels in loader:

            # Move input texts and labels to the designated device.
            texts = texts.to(device)
            labels = labels.to(device)

            # Forward pass to get raw outputs from the model.
            outputs = model(texts)

            # Apply sigmoid to convert logits to probabilities (0 to 1).
            predictions = torch.sigmoid(outputs)

            # Convert probabilities to binary predictions (0 or 1) based on a 0.5 threshold.
            predicted = (predictions >= 0.5).float()

            # Count the number of correct predictions.
            correct += (
                predicted == labels
            ).sum().item()

            # Add the total number of labels in the current batch to the total count.
            total += labels.size(0)

    # Calculate the overall accuracy.
    accuracy = correct / total

    return accuracy

# ============================================================
# TEST ACCURACY
# ============================================================

# Calculate and print the model's accuracy on the test dataset.
accuracy = evaluate(model, test_loader)

print("Test Accuracy:", accuracy)

# ============================================================
# PREDICT CUSTOM SENTENCE
# ============================================================

def predict_sentiment(sentence):
    # Function to predict the sentiment of a single custom sentence.

    # Set the model to evaluation mode.
    model.eval()

    # Encode the input sentence into numerical indices.
    encoded = encode(sentence)

    # Convert the encoded list to a PyTorch tensor, add a batch dimension (unsqueeze(0)),
    # and move it to the correct device (CPU/GPU).
    tensor = torch.tensor(
        encoded,
        dtype=torch.long
    ).unsqueeze(0).to(device)

    # Disable gradient calculations for prediction.
    with torch.no_grad():

        # Get the raw output (logits) from the model.
        output = model(tensor)

        # Apply sigmoid to convert the logit to a probability.
        prediction = torch.sigmoid(output)

    # Determine sentiment based on the probability threshold.
    if prediction >= 0.5:

        sentiment = "Positive"

    else:

        sentiment = "Negative"

    # Print the original review and its predicted sentiment.
    print("Review:", sentence)

    print("Sentiment:", sentiment)

# ============================================================
# TEST SENTENCES
# ============================================================

# Test the `predict_sentiment` function with example sentences.
predict_sentiment(
    "This movie was amazing and wonderful"
)

predict_sentiment(
    "The film was boring and terrible"
)


Using Device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


{'text': 'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far be

More text sentences

In [3]:
# ============================================================
# More Complex TEST SENTENCES
# ============================================================

predict_sentiment(
    "This movie was amazing and wonderful though confusing to watch"
)

predict_sentiment(
    "The film was unexpected, boring, slow paced, I would definitely watch again"
)

Review: This movie was amazing and wonderful though confusing to watch
Sentiment: Negative
Review: The film was unexpected, boring, slow paced, I would definitely watch again
Sentiment: Positive


# **Discussion**
It performs better with simple sentences.
Though, with more complex and intentionallymisleadling sentences it struggles.

FOr instance for "This movie was amazing and wonderful though confusing to watch"--> It predicts "Negative". Even though the review is mostly positive